# HoloSyn: Multimodal P2P Co‑Regulation — Open‑Source Hugging Face Teacher Distillation (Colab)

Runs locally using **open-source Hugging Face models** (no API keys).

Pipeline:
1. Extract audio/video/image/text/haptics from `Archive.zip`
2. Run **teacher inference** with open-source models
3. Map teacher outputs → targets: `valence, arousal, calm, trust` (0..1)
4. Train a compact **student head** and export **TorchScript**
5. Optional runtime: **Cirq/qsimcirq** synchrony + **Brian2** entrainment


In [2]:
#@title 0) Install dependencies
!pip -q install -U numpy pandas tqdm pillow opencv-python soundfile librosa
!pip -q install -U torch torchvision torchaudio
!pip -q install -U transformers accelerate sentencepiece safetensors
!pip -q install -U cirq qsimcirq brian2

import os, json, zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import torch
print("✅ Installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 59.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cirq-core 1.6.1 requires pandas~=2.1, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.1.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [4]:
#@title 1) Locate and extract Archive.zip
def pick_existing(*cands):
    for c in cands:
        if c and os.path.exists(c):
            return c
    return None

ARCHIVE_ZIP = pick_existing("/mnt/data/Archive.zip", "/content/Archive.zip")
assert ARCHIVE_ZIP, "❌ Archive.zip not found. Upload it to Colab Files or place in /mnt/data."

EXTRACT_DIR = "/content/archive_extracted"
Path(EXTRACT_DIR).mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVE_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

print("ARCHIVE_ZIP:", ARCHIVE_ZIP)
print("EXTRACT_DIR:", EXTRACT_DIR)
print("Top-level:")
for p in sorted(Path(EXTRACT_DIR).iterdir())[:60]:
    print(" -", p.name, "(dir)" if p.is_dir() else "(file)")

ARCHIVE_ZIP: /content/Archive.zip
EXTRACT_DIR: /content/archive_extracted
Top-level:
 - E_aud.npy (file)
 - E_img.npy (file)
 - E_text.npy (file)
 - E_vid.npy (file)
 - audio (dir)
 - haptics (dir)
 - images (dir)
 - manifest.jsonl (file)
 - text (dir)
 - video (dir)


In [5]:
#@title 2) Discover files by modality
AUDIO_EXT = {".wav",".mp3",".m4a",".flac",".ogg",".aac"}
VIDEO_EXT = {".mp4",".mov",".mkv",".webm",".avi"}
IMAGE_EXT = {".png",".jpg",".jpeg",".webp",".bmp"}
TEXT_EXT  = {".txt",".md",".json",".csv",".tsv"}

def walk_files(root):
    out=[]
    for p in Path(root).rglob("*"):
        if p.is_file(): out.append(str(p))
    return out

def bucket(path):
    ext = Path(path).suffix.lower()
    low = path.lower()
    if ext in AUDIO_EXT: return "audio"
    if ext in VIDEO_EXT: return "video"
    if ext in IMAGE_EXT: return "image"
    if ext in TEXT_EXT:
        if ext in {".json",".csv",".tsv"} and any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
            return "haptics"
        return "text"
    if any(k in low for k in ["hapt","vib","pattern","tact","rumble"]):
        return "haptics"
    return "other"

all_files = walk_files(EXTRACT_DIR)
buckets={}
for f in all_files:
    buckets.setdefault(bucket(f), []).append(f)

for k in ["audio","video","image","text","haptics","other"]:
    print(f"{k:8s}", len(buckets.get(k, [])))
    for s in buckets.get(k, [])[:3]:
        print("   •", s.replace(EXTRACT_DIR + "/", ""))

audio    49
   • audio/model_c000002_0000004.wav
   • audio/cookie_c000003_0000025.wav
   • audio/cookie_c000002_0000012.wav
video    50
   • video/cookie_c000003_0000024.mp4
   • video/model_c000002_0000006.mp4
   • video/model_c000003_0000033.mp4
image    115
   • images/kenzonaplane-20260216-0052.jpg
   • images/client_c000002_0000015.png
   • images/kenzonaplane-20260216-0059.jpg
text     49
   • text/client_c000002_0000017.txt
   • text/model_c000002_0000004.txt
   • text/blockheart_c000001_0000048.txt
haptics  67
   • haptics/0007.npy
   • haptics/0001.npy
   • haptics/0034.npy
other    5
   • E_vid.npy
   • E_text.npy
   • manifest.jsonl


In [6]:
#@title 3) Local features (privacy-preserving)
import librosa, cv2

def rel(p): return p.replace(EXTRACT_DIR + "/", "")
def load_text(path, max_chars=12000):
    return open(path, "r", encoding="utf-8", errors="ignore").read()[:max_chars]

def load_haptics_any(path):
    ext = Path(path).suffix.lower()
    if ext == ".json":
        try: return json.load(open(path, "r", encoding="utf-8", errors="ignore"))
        except: return {"raw": load_text(path)}
    if ext in {".csv",".tsv"}:
        sep = "," if ext==".csv" else "\t"
        try:
            df = pd.read_csv(path, sep=sep)
            return {"columns": list(df.columns), "head": df.head(200).to_dict(orient="list")}
        except:
            return {"raw": load_text(path)}
    return {"raw": load_text(path)}

def featurize_text(s):
    return {"txt_len":float(len(s)),
            "txt_lines":float(s.count("\n")+1),
            "txt_exclaim":float(s.count("!")),
            "txt_question":float(s.count("?")),
            "txt_caps_ratio":float(sum(c.isupper() for c in s)/max(1,len(s)))}

def featurize_haptics(h):
    raw = json.dumps(h)[:20000].lower()
    return {"hapt_len":float(len(raw)),
            "hapt_has_intensity":1.0 if "intensity" in raw else 0.0,
            "hapt_has_freq":1.0 if ("hz" in raw or "freq" in raw) else 0.0}

def audio_features(path, sr=16000, max_seconds=15):
    y, _ = librosa.load(path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return {"aud_rms":0.0,"aud_zcr":0.0,"aud_centroid":0.0,"aud_tempo":0.0}
    rms = float(np.mean(librosa.feature.rms(y=y)))
    zcr = float(np.mean(librosa.feature.zero_crossing_rate(y)))
    centroid = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    onset_env = librosa.onset.onset_strength(y=y, sr=sr)
    tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
    return {"aud_rms":rms,"aud_zcr":zcr,"aud_centroid":centroid,"aud_tempo":tempo}

def image_quick_stats(path):
    im = Image.open(path).convert("RGB")
    arr = np.asarray(im).astype(np.float32)/255.0
    mean = arr.mean(axis=(0,1))
    std = arr.std(axis=(0,1))
    return {"img_w":float(arr.shape[1]),"img_h":float(arr.shape[0]),
            "img_mean_r":float(mean[0]),"img_mean_g":float(mean[1]),"img_mean_b":float(mean[2]),
            "img_std_r":float(std[0]),"img_std_g":float(std[1]),"img_std_b":float(std[2])}

def sample_video_frames(video_path, every_n_frames=45, max_frames=4, target_size=224):
    cap = cv2.VideoCapture(video_path)
    frames=[]
    idx=0
    while cap.isOpened() and len(frames) < max_frames:
        ret, frame = cap.read()
        if not ret: break
        if idx % every_n_frames == 0:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            h,w = frame.shape[:2]
            if max(h,w) > target_size:
                scale = target_size / max(h,w)
                frame = cv2.resize(frame, (int(w*scale), int(h*scale)))
            frames.append(frame)
        idx += 1
    cap.release()
    return frames

MAX_PER_MODALITY = 250
rows=[]
for p in tqdm(buckets.get("text", [])[:MAX_PER_MODALITY], desc="text"):
    rows.append({"path":p,"rel":rel(p),"modality":"text",**featurize_text(load_text(p))})
for p in tqdm(buckets.get("haptics", [])[:MAX_PER_MODALITY], desc="haptics"):
    rows.append({"path":p,"rel":rel(p),"modality":"haptics",**featurize_haptics(load_haptics_any(p))})
for p in tqdm(buckets.get("audio", [])[:MAX_PER_MODALITY], desc="audio"):
    rows.append({"path":p,"rel":rel(p),"modality":"audio",**audio_features(p)})
for p in tqdm(buckets.get("image", [])[:MAX_PER_MODALITY], desc="image"):
    try: rows.append({"path":p,"rel":rel(p),"modality":"image",**image_quick_stats(p)})
    except Exception as e: print("skip image", rel(p), e)
for p in tqdm(buckets.get("video", [])[:MAX_PER_MODALITY], desc="video"):
    try: rows.append({"path":p,"rel":rel(p),"modality":"video","vid_n_frames":float(len(sample_video_frames(p)))})
    except Exception as e: print("skip video", rel(p), e)

df = pd.DataFrame(rows).fillna(0.0)
print("✅ Base features:", df.shape)
df.head()

audio:   0%|          | 0/49 [00:00<?, ?it/s]/tmp/ipython-input-2621360028.py:43: FutureWarning: librosa.beat.tempo
	This function was moved to 'librosa.feature.rhythm.tempo' in librosa version 0.10.0.
	This alias will be removed in librosa version 1.0.
  tempo = float(librosa.beat.tempo(onset_envelope=onset_env, sr=sr)[0]) if onset_env.size else 0.0
video: 100%|██████████| 50/50 [00:00<00:00, 50.90it/s]

✅ Base features: (330, 24)


,path,rel,modality,txt_len,txt_lines,txt_exclaim,txt_question,txt_caps_ratio,hapt_len,hapt_has_intensity,...,aud_tempo,img_w,img_h,img_mean_r,img_mean_g,img_mean_b,img_std_r,img_std_g,img_std_b,vid_n_frames
0,/content/archive_extracted/text/client_c000002...,text/client_c000002_0000017.txt,text,104.0,1.0,0.0,0.0,0.057692,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,/content/archive_extracted/text/model_c000002_...,text/model_c000002_0000004.txt,text,103.0,1.0,0.0,0.0,0.058252,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,/content/archive_extracted/text/blockheart_c00...,text/blockheart_c000001_0000048.txt,text,106.0,1.0,0.0,0.0,0.056604,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,/content/archive_extracted/text/model_c000004_...,text/model_c000004_0000036.txt,text,101.0,1.0,0.0,0.0,0.059406,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,/content/archive_extracted/text/model_c000004_...,text/model_c000004_0000034.txt,text,103.0,1.0,0.0,0.0,0.058252,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [9]:
#@title 4) Load open-source teachers (HF)
!pip install -q -U pillow==9.5.0
from transformers import pipeline, AutoProcessor, AutoModel
import torch

device = 0 if torch.cuda.is_available() else -1
print("device:", "cuda" if device==0 else "cpu")

EMO_MODEL = "j-hartmann/emotion-english-distilroberta-base"
SENT_MODEL = "cardiffnlp/twitter-roberta-base-sentiment-latest"
CLIP_MODEL = "openai/clip-vit-base-patch32"
W2V_MODEL  = "facebook/wav2vec2-base-960h"

emo_pipe = pipeline("text-classification", model=EMO_MODEL, top_k=None, device=device)
sent_pipe = pipeline("text-classification", model=SENT_MODEL, top_k=None, device=device)

clip_processor = AutoProcessor.from_pretrained(CLIP_MODEL)
clip_model = AutoModel.from_pretrained(CLIP_MODEL).to("cuda" if device==0 else "cpu").eval()

w2v_processor = AutoProcessor.from_pretrained(W2V_MODEL)
w2v_model = AutoModel.from_pretrained(W2V_MODEL).to("cuda" if device==0 else "cpu").eval()

print("✅ Teachers loaded")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 MB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
scikit-image 0.25.2 requires pillow>=10.1, but you have pillow 9.5.0 which is incompatible.
device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/329M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/329M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/378M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/210 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base-960h
Key               | Status     | 
------------------+------------+-
lm_head.weight    | UNEXPECTED | 
lm_head.bias      | UNEXPECTED | 
masked_spec_embed | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Teachers loaded


In [10]:
#@title 5) Teacher cache: targets + embeddings
def softmax_dict(items):
    d = {it["label"].lower(): float(it["score"]) for it in items}
    s = sum(d.values()) or 1.0
    return {k:v/s for k,v in d.items()}

def emotion_to_targets(emo_probs, sent_probs):
    valence = sent_probs.get("positive",0.33) + 0.5*sent_probs.get("neutral",0.33)
    valence = float(np.clip(valence,0,1))
    arousal = (
        1.0*emo_probs.get("anger",0) +
        1.0*emo_probs.get("fear",0) +
        0.8*emo_probs.get("surprise",0) +
        0.6*emo_probs.get("joy",0) +
        0.2*emo_probs.get("sadness",0) +
        0.3*emo_probs.get("disgust",0)
    )
    arousal = float(np.clip(arousal,0,1))
    calm = float(np.clip(1.0-arousal,0,1))
    trust = float(np.clip(valence*(1.0-0.8*emo_probs.get("fear",0)-0.6*emo_probs.get("anger",0)),0,1))
    return valence, arousal, calm, trust

@torch.no_grad()
def clip_image_embedding(pil_images):
    if not isinstance(pil_images, list): pil_images=[pil_images]
    inputs = clip_processor(images=pil_images, return_tensors="pt")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k:v.to(dev) for k,v in inputs.items()}
    out = clip_model.get_image_features(**inputs)
    out = out / out.norm(dim=-1, keepdim=True)
    return out.cpu().numpy()

@torch.no_grad()
def wav2vec_embedding(wav_path, sr=16000, max_seconds=10):
    y, _ = librosa.load(wav_path, sr=sr, mono=True, duration=max_seconds)
    if y.size < 10:
        return np.zeros((768,), dtype=np.float32)
    inputs = w2v_processor(y, sampling_rate=sr, return_tensors="pt")
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = {k:v.to(dev) for k,v in inputs.items()}
    out = w2v_model(**inputs).last_hidden_state
    emb = out.mean(dim=1).squeeze(0)
    emb = emb / (emb.norm() + 1e-8)
    return emb.detach().cpu().numpy().astype(np.float32)

CACHE_PATH="/content/hf_teacher_cache.parquet"
teacher_df = pd.read_parquet(CACHE_PATH) if os.path.exists(CACHE_PATH) else pd.DataFrame()
done=set(teacher_df["rel"].tolist()) if len(teacher_df) else set()

records=[]
N_PER_MODALITY=180

# text/haptics targets
for modality in ["text","haptics"]:
    sub = df[df["modality"]==modality].head(N_PER_MODALITY)
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"teacher {modality}"):
        if row["rel"] in done: continue
        try:
            txt = load_text(row["path"]) if modality=="text" else ("HAPTICS_JSON:\n"+json.dumps(load_haptics_any(row["path"]))[:12000])
            emo = emo_pipe(txt[:3000])[0]
            sent = sent_pipe(txt[:3000])[0]
            emo_probs, sent_probs = softmax_dict(emo), softmax_dict(sent)
            v,a,c,t = emotion_to_targets(emo_probs, sent_probs)
            records.append({"rel":row["rel"],"modality":modality,"valence":v,"arousal":a,"calm":c,"trust":t})
        except Exception as e:
            print("skip", row["rel"], e)

# image/video clip embeddings
for modality in ["image","video"]:
    sub = df[df["modality"]==modality].head(N_PER_MODALITY)
    for _, row in tqdm(sub.iterrows(), total=len(sub), desc=f"teacher {modality}"):
        if row["rel"] in done: continue
        try:
            if modality=="image":
                ims=[Image.open(row["path"]).convert("RGB")]
            else:
                frames=sample_video_frames(row["path"])
                if not frames: continue
                ims=[Image.fromarray(f).convert("RGB") for f in frames]
            embs=clip_image_embedding(ims)
            emb=embs.mean(axis=0)
            emb=emb/(np.linalg.norm(emb)+1e-8)
            records.append({"rel":row["rel"],"modality":modality,
                            "valence":0.5,"arousal":0.5,"calm":0.5,"trust":0.5,
                            "clip_emb":emb.tolist()})
        except Exception as e:
            print("skip", modality, row["rel"], e)

# audio wav2vec embedding
sub = df[df["modality"]=="audio"].head(N_PER_MODALITY)
for _, row in tqdm(sub.iterrows(), total=len(sub), desc="teacher audio"):
    if row["rel"] in done: continue
    try:
        emb=wav2vec_embedding(row["path"])
        records.append({"rel":row["rel"],"modality":"audio",
                        "valence":0.5,"arousal":0.5,"calm":0.5,"trust":0.5,
                        "w2v_emb":emb.tolist()})
    except Exception as e:
        print("skip audio", row["rel"], e)

new_df=pd.DataFrame(records)
teacher_df = pd.concat([teacher_df, new_df], ignore_index=True) if len(teacher_df) else new_df
teacher_df.to_parquet(CACHE_PATH, index=False)
print("✅ Teacher cache:", teacher_df.shape, "saved to", CACHE_PATH)
teacher_df.head()

teacher haptics:  24%|██▍       | 16/67 [00:00<00:00, 88.73it/s]

skip haptics/0007.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0001.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0034.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0014.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0020.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/tmplfehff5d.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0045.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/tmpq3e7wy9g.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0038.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0027.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0000.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0006.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0029.npy index 514 is out

teacher haptics:  70%|███████   | 47/67 [00:00<00:00, 130.74it/s]

skip haptics/0047.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0005.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0044.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0035.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0010.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0048.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0030.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0026.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/tmp9rqdlj5p.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0043.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/tmp9df1uhrn.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/tmptxjm61l_.npy index 514 is out of bounds for dimension 1 with size 514
skip haptics/0021.npy index 514

teacher haptics: 100%|██████████| 67/67 [00:00<00:00, 119.70it/s]


...

teacher video: 100%|██████████| 50/50 [00:13<00:00,  3.67it/s]


skip video video/cookie_c000004_0000044.mp4 'BaseModelOutputWithPooling' object has no attribute 'norm'


teacher audio: 100%|██████████| 49/49 [00:17<00:00,  2.87it/s]

✅ Teacher cache: (98, 7) saved to /content/hf_teacher_cache.parquet


,rel,modality,valence,arousal,calm,trust,w2v_emb
0,text/client_c000002_0000017.txt,text,0.511199,0.150625,0.849375,0.507465,NaN
1,text/model_c000002_0000004.txt,text,0.508652,0.198813,0.801187,0.505504,NaN
2,text/blockheart_c000001_0000048.txt,text,0.518708,0.059151,0.940849,0.514630,NaN
3,text/model_c000004_0000036.txt,text,0.513505,0.057568,0.942432,0.508758,NaN
4,text/model_c000004_0000034.txt,text,0.510257,0.167567,0.832433,0.506945,NaN


In [16]:
#@title 6) Train student head + export TorchScript
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

merged = df.merge(pd.read_parquet(CACHE_PATH), on=["rel","modality"], how="inner")

# expand embeddings
def expand_vec(colname, prefix):
    if colname not in merged.columns: return
    vecs = merged[colname].dropna()
    if len(vecs)==0: return
    dim = len(vecs.iloc[0])

    # The original error was because fillna was called with a list, which is not supported for Series.
    # Instead, we iterate and conditionally replace non-list items (including NaN) with zero vectors.
    arr_list = []
    for val in merged[colname]:
        if isinstance(val, list):
            arr_list.append(val)
        else:
            arr_list.append([0.0]*dim)

    arr = np.vstack(arr_list)

    for i in range(dim):
        merged[f"{prefix}{i}"] = arr[:, i].astype(np.float32)
    merged.drop(columns=[colname], inplace=True)

expand_vec("clip_emb","clip_")
expand_vec("w2v_emb","w2v_")

TARGETS=["valence","arousal","calm","trust"]
ignore=set(["path","rel","modality"]+TARGETS)
numeric_cols=[c for c in merged.columns if c not in ignore and pd.api.types.is_numeric_dtype(merged[c])]

X = merged[numeric_cols].astype(np.float32).values
Y = merged[TARGETS].astype(np.float32).values

mu=X.mean(axis=0, keepdims=True)
sd=X.std(axis=0, keepdims=True)+1e-6
Xn=(X-mu)/sd

class TableDS(Dataset):
    def __init__(self, X, Y):
        self.X=torch.tensor(X, dtype=torch.float32)
        self.Y=torch.tensor(Y, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self,i): return self.X[i], self.Y[i]

ds=TableDS(Xn,Y)
dl=DataLoader(ds,batch_size=64,shuffle=True)

student = nn.Sequential(
    nn.Linear(Xn.shape[1],256), nn.Tanh(),
    nn.Linear(256,128), nn.Tanh(),
    nn.Linear(128,4), nn.Sigmoid()
)

dev="cuda" if torch.cuda.is_available() else "cpu"
student.to(dev)
opt=torch.optim.Adam(student.parameters(), lr=1e-3)
loss_fn=nn.MSELoss()

student.train()
for epoch in range(12000):
    total=0.0
    for xb,yb in dl:
        xb,yb=xb.to(dev), yb.to(dev)
        pred=student(xb)
        loss=loss_fn(pred,yb)
        opt.zero_grad(); loss.backward(); opt.step()
        total += loss.item()*xb.size(0)
    print(f"epoch {epoch:02d} mse {total/len(ds):.6f}")

student_cpu = student.to("cpu").eval()
example=torch.zeros(1, Xn.shape[1])
ts=torch.jit.trace(student_cpu, example)
student_ts_path="/content/student_distilled_heads_hf.torchscript.pt"
ts.save(student_ts_path)

norm_path="/content/student_norm_hf.json"
with open(norm_path,"w",encoding="utf-8") as f:
    json.dump({"numeric_cols":numeric_cols,"mu":mu.flatten().tolist(),"sd":sd.flatten().tolist()}, f)

print("✅ Saved:", student_ts_path)
print("✅ Norm :", norm_path)


WARNING    /tmp/ipython-input-749115814.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[f"{prefix}{i}"] = arr[:, i].astype(np.float32)
 [py.warnings]
  merged[f"{prefix}{i}"] = arr[:, i].astype(np.float32)

WARNING    /tmp/ipython-input-749115814.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged[f"{prefix}{i}"] = arr[:, i].astype(np.float32)
 [py.warnings]
  merged[f"{prefix}{i}"] = arr[:, i].astype(np.float32)

WARNING    /tmp/ipython-input-749115814.py:26: PerformanceWarning: DataFrame is highly fragmente

Streaming output truncated to the last 5000 lines.
epoch 7002 mse 0.000568
epoch 7003 mse 0.000567
epoch 7004 mse 0.000567
epoch 7005 mse 0.000569
epoch 7006 mse 0.000574
epoch 7007 mse 0.000566
epoch 7008 mse 0.000570
epoch 7009 mse 0.000588
epoch 7010 mse 0.000601
epoch 7011 mse 0.000579
epoch 7012 mse 0.000572
epoch 7013 mse 0.000615
epoch 7014 mse 0.000572
epoch 7015 mse 0.000578
epoch 7016 mse 0.000575
epoch 7017 mse 0.000565
epoch 7018 mse 0.000568
epoch 7019 mse 0.000563
epoch 7020 mse 0.000562
epoch 7021 mse 0.000565
epoch 7022 mse 0.000564
epoch 7023 mse 0.000587
epoch 7024 mse 0.000583
epoch 7025 mse 0.000568
epoch 7026 mse 0.000586
epoch 7027 mse 0.000591
epoch 7028 mse 0.000593
epoch 7029 mse 0.000576
epoch 7030 mse 0.000591
epoch 7031 mse 0.000591
epoch 7032 mse 0.000581
epoch 7033 mse 0.000559
epoch 7034 mse 0.000588
epoch 7035 mse 0.000582
epoch 7036 mse 0.000577
epoch 7037 mse 0.000583
epoch 7038 mse 0.000575
epoch 7039 mse 0.000585
epoch 7040 mse 0.000573
epoch 7041 ms

In [23]:
#@title 7) Optional: Quantum synchrony + Brian2 entrainment
import cirq, qsimcirq
from brian2 import *

class QuantumSynchrony:
    def __init__(self, n_qubits=4, depth=2):
        self.n_qubits=n_qubits; self.depth=depth
        self.qubits=cirq.LineQubit.range(n_qubits)
        self.sim=qsimcirq.QSimSimulator()
    def _reduce(self, e):
        x=np.tanh(np.asarray(e,dtype=np.float32))*np.pi
        x=x[:self.n_qubits]
        if x.size<self.n_qubits: x=np.pad(x,(0,self.n_qubits-x.size))
        return x
    def _circuit(self, x):
        c=cirq.Circuit()
        c.append([cirq.H(q) for q in self.qubits])
        for _ in range(self.depth):
            for i,q in enumerate(self.qubits):
                c.append(cirq.ry(x[i])(q)); c.append(cirq.rz(0.5*x[i])(q))
            for i in range(self.n_qubits-1):
                c.append(cirq.CZ(self.qubits[i], self.qubits[i+1]))
        return c
    def kernel(self, ea, eb):
        sa=self.sim.simulate(self._circuit(self._reduce(ea))).final_state_vector
        sb=self.sim.simulate(self._circuit(self._reduce(eb))).final_state_vector
        return float(np.abs(np.vdot(sa,sb))**2)

def plv_from_coupling(coupling=0.2, duration_ms=200):
    start_scope()
    eqs='''dphi/dt = omega + k*sin(phi_other - phi) : Hz
    phi_other : 1
    omega : Hz
    k : Hz'''
    G=NeuronGroup(2, eqs, method='euler')
    # G.phi should be unitless for phase, G.omega must be Hz as declared in eqs.
    G.phi=[0.1,2.0]; G.omega=[2*np.pi*8*Hz,2*np.pi*8*Hz]; G.k=coupling * Hz
    @network_operation(dt=1*ms)
    def couple():
        G.phi_other[0]=G.phi[1]; G.phi_other[1]=G.phi[0]
    M=StateMonitor(G,'phi',record=True)
    net=Network(G,couple,M); net.run(duration_ms*ms)
    phase_diff=np.array(M.phi[0]-M.phi[1])
    return float(np.abs(np.mean(np.exp(1j*phase_diff))))

qs=QuantumSynchrony()
print("qsync demo:", qs.kernel([0.2,0.1,0.6,0.4],[0.2,0.1,0.6,0.4]))
print("plv demo:", plv_from_coupling(0.05), plv_from_coupling(0.35))


qsync demo: 1.0000004768371582


DimensionMismatchError: phi should be set with a value with units hertz, but got [0.1 2. ] (unit is 1).

In [26]:
import torch, json
import numpy as np

MODEL_PATH = "/content/student_distilled_heads_hf.torchscript.pt"
NORM_PATH  = "/content/student_norm_hf.json"

model = torch.jit.load(MODEL_PATH)
model.eval()

with open(NORM_PATH) as f:
    norm = json.load(f)

mu = np.array(norm["mu"], dtype=np.float32)
sd = np.array(norm["sd"], dtype=np.float32)

def normalize(x):
    return (x - mu) / sd

def predict(feature_vector):
    x = normalize(np.array(feature_vector, dtype=np.float32))
    x = torch.tensor(x).unsqueeze(0)
    with torch.no_grad():
        y = model(x)
    return y.squeeze().numpy()